# Prepare Imaging Data using CLAM

To obtain the most informative patches from the Whole Slide Images (WSIs) we use Clustering-constrained Attention Multiple Instance Learning (CLAM), which is a deep learning framework tailored for weakly-supervised classification of WSIs in computational pathology. Based on the Multiple Instance Learning (MIL) paradigm, CLAM treats each WSI as a bag of image patches and uses attention-based mechanisms to identify the most diagnostically relevant regions without requiring pixel-level annotations.

To improve robustness, CLAM incorporates instance-level clustering constraints on high- and low-attention patches, refining the feature space and reducing noise. One of its key strengths is interpretability—attention heatmaps highlight the regions contributing most to the model’s decisions, making results transparent and insightful for pathologists. Overall, CLAM enables efficient, accurate, and interpretable WSI-level classification, even in the absence of detailed annotations.

See the [paper](https://www.nature.com/articles/s41551-020-00682-w) (Data-efficient and weakly supervised computational pathology on whole-slide images)

<img src="../data_download/Fig/patch_extraction.jpg" alt="Descripción de la imagen" width="1000"/>

## Installation

### Create Conda Environment from [`env.yml`](../CLAM/env.yml)

To set up the environment required by CLAM, run the following command in your terminal:

```bash
conda env create -f ../CLAM/env.yml
```

For detailed installation instructions, please refer to the [CLAM Installation Guide](https://github.com/mahmoodlab/CLAM/blob/master/docs/INSTALLATION.md).


## WSI Segmentation and Patching

 

To prepare WSIs for use with CLAM, a **segmentation and patching** step is required. This process identifies tissue regions within each slide and extracts smaller image patches for feature extraction and model training. For more details, see the official [CLAM GitHub repository](https://github.com/mahmoodlab/CLAM#installation). The `tcga.csv` file provides a predefined segmentation preset optimized for TCGA slides.

You can perform segmentation and patch extraction using the following command:

```bash
python create_patches_fp.py --source DATA_DIRECTORY --save_dir RESULTS_DIRECTORY --patch_size 256 --preset tcga.csv --seg --patch 
```

This command will:

* Segment tissue regions from the WSIs,
* Extract 256×256 pixel patches only from tissue areas,
* Save the resulting patches in the specified results directory.

 


### Weakly-Supervised Learning using Slide-Level Labels with CLAM
 After segmenting the slides and generating patches, the next step is to extract features from each patch using a pretrained CNN (We use ResNet50). These features are later used for training the attention-based MIL model.
 
The following command runs the feature extraction step using the fully patched slides:

```bash
CUDA_VISIBLE_DEVICES=0 python extract_features_fp.py \
  --data_h5_dir DIR_TO_COORDS \
  --data_slide_dir DATA_DIRECTORY \
  --csv_path CSV_FILE_NAME \
  --feat_dir FEATURES_DIRECTORY \
  --batch_size 512 \
  --slide_ext .svs
```

* `--data_h5_dir`: Directory containing `.h5` coordinate files from the patching step
* `--data_slide_dir`: Directory with original WSI files (e.g., `.svs`)
* `--csv_path`: Path to the slide metadata CSV (used for matching slides)
* `--feat_dir`: Directory where extracted features will be saved
* `--slide_ext`: Extension of the slide files (commonly `.svs` for TCGA)

For more details, refer to the [CLAM GitHub repository](https://github.com/mahmoodlab/CLAM#installation).
 

### Heatmap Visualization

 
CLAM provides support for generating attention-based heatmaps that highlight the most diagnostically relevant regions within each WSI. These visualizations can help interpret model decisions by showing which patches received the highest attention scores. Heatmaps are created using the following command:

```bash
CUDA_VISIBLE_DEVICES=0 python create_heatmaps.py --config config_template.yaml
```

Although heatmap visualization was explored during the development of our pipeline, it was ultimately not included in the SAMVAE framework. 

### Explored Imaging Scenarios for LGG and BRCA

The following figure illustrates several scenarios that were studied for processing imaging data from LGG (left) and BRCA (right):
a) and b) show WSIs downscaled to 128×128 pixels for rapid visualization;
c) and d) depict tissue segmentation using the CLAM pipeline;
e) and f) present attention-based heatmaps highlighting diagnostically relevant regions;
finally, g) and h) display high-attention score patches extracted from these heatmaps.

After evaluating all these approaches, we ultimately selected the strategy of directly using the top informative patches, as it provided better performance for our SAMVAE framework.
 

<img src="../data_download/Fig/wsis_scenarios.png" alt="Descripción de la imagen" width="1000"/>